# StatPitch
## Notebook 8 — Transfermarkt Squad Values

---

Adds three new features per team to `team_stats.json`:

| Feature | What it measures | Why it matters |
|---|---|---|
| `squad_value_eur` | Total squad market value (€) | Overall talent concentration |
| `avg_player_value_eur` | Mean value per player | Squad depth vs star-heavy |
| `squad_age` | Average squad age | Peak vs rebuilding phase |

**Strategy:** Transfermarkt blocks most scrapers. We use three layers of defence:
1. `cloudscraper` to bypass Cloudflare
2. Realistic browser headers and random delays
3. A cached JSON file — once a team is scraped, it's never re-fetched

> Squad values change slowly (once per transfer window). Re-run this notebook once every 3–4 months, not before every match.

---
## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

SAVE_DIR    = Path('/content/drive/MyDrive/StatPitch')
PROJECT_DIR = SAVE_DIR / 'statpitch_api'
os.chdir(SAVE_DIR)

!pip install cloudscraper --quiet

import cloudscraper
import json, time, re, shutil
import random
from bs4 import BeautifulSoup
import pandas as pd

print('Setup complete — working in:', os.getcwd())
CACHE_FILE = Path('/content/drive/MyDrive/StatPitch/tm_squad_cache.json')
CACHE_FILE.unlink(missing_ok=True)
print('Cache cleared — will re-scrape all teams with fixed parser')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete — working in: /content/drive/MyDrive/StatPitch
Cache cleared — will re-scrape all teams with fixed parser


---
## Step 1 — Team URL map

Transfermarkt URLs use internal team IDs. We maintain a manual map of the
most important national teams. Add more as needed.

In [ ]:
# Transfermarkt national team URLs
# Format: team_name (must match team_stats.json) -> TM slug/id
TM_TEAMS = {
    # HOSTS
    'Canada':               'kanada/startseite/verein/3522',
    'Mexico':               'mexiko/startseite/verein/3424',
    'United States':        'vereinigte-staaten/startseite/verein/3425',

    # CONMEBOL
    'Argentina':            'argentinien/startseite/verein/3437',
    'Brazil':               'brasilien/startseite/verein/3439',
    'Colombia':             'kolumbien/startseite/verein/3441',
    'Ecuador':              'ecuador/startseite/verein/3447',
    'Paraguay':             'paraguay/startseite/verein/3442',
    'Uruguay':              'uruguay/startseite/verein/3444',

    # UEFA
    'Austria':              'oesterreich/startseite/verein/3385',
    'Belgium':              'belgien/startseite/verein/3382',
    'Bosnia-Herzegovina':   'bosnien-herzegowina/startseite/verein/3557',
    'Croatia':              'kroatien/startseite/verein/3556',
    'Czech Republic': 'tschechische-republik/startseite/verein/3398',
    'England':        'england/startseite/verein/3411/saison_id/2025',
    'France':               'frankreich/startseite/verein/3377',
    'Germany':              'deutschland/startseite/verein/3376',
    'Netherlands':          'niederlande/startseite/verein/3378',
    'Norway':               'norwegen/startseite/verein/3383',
    'Portugal':             'portugal/startseite/verein/3380',
    'Scotland':             'schottland/startseite/verein/3410',
    'Spain':                'spanien/startseite/verein/3375',
    'Sweden':         'schweden/startseite/verein/3389/saison_id/2025',
    'Switzerland':          'schweiz/startseite/verein/3386',
    'Turkey':               'tuerkei/startseite/verein/3388',

    # CAF
    'Algeria':        'algerien/startseite/verein/3421/saison_id/2025',
    'Cape Verde':           'kap-verde/startseite/verein/3526',
    'DR Congo':             'dr-kongo/startseite/verein/3523',
    'Ivory Coast':          'elfenbeinkueste/startseite/verein/3416',
    'Egypt':                'aegypten/startseite/verein/3418',
    'Ghana':                'ghana/startseite/verein/3419',
    'Morocco':              'marokko/startseite/verein/3413',
    'Senegal':              'senegal/startseite/verein/3414',
    'South Africa':         'suedafrika/startseite/verein/3524',
    'Tunisia':        'tunesien/startseite/verein/3420/saison_id/2025',

    # AFC
    'Australia':            'australien/startseite/verein/3432',
    'Iran':                 'iran/startseite/verein/3434',
    'Iraq':           'irak/startseite/verein/3531/saison_id/2025',
    'Japan':                'japan/startseite/verein/3435',
    'Jordan':               'jordanien/startseite/verein/3532',
    'Qatar':                'katar/startseite/verein/3533',
    'Saudi Arabia':         'saudi-arabien/startseite/verein/3433',
    'South Korea':          'suedkorea/startseite/verein/3436',
    'Uzbekistan':     'usbekistan/startseite/verein/3534/saison_id/2025',

    # OFC
    'New Zealand':    'neuseeland/startseite/verein/3430/saison_id/2025',

    # CONCACAF non-hosts
    'Curacao':              'curacao/startseite/verein/3535',
    'Haiti':                'haiti/startseite/verein/3428',
    'Panama':               'panama/startseite/verein/3429',
}

TM_BASE = 'https://www.transfermarkt.com'
CACHE_FILE = SAVE_DIR / 'tm_squad_cache.json'

print(f'{len(TM_TEAMS)} national teams configured')

48 national teams configured


---
## Step 2 — Scraper setup and cache

In [ ]:
# Load existing cache (so we don't re-scrape teams we already have)
if CACHE_FILE.exists():
    with open(CACHE_FILE) as f:
        cache = json.load(f)
    print(f'Cache loaded — {len(cache)} teams already scraped')
else:
    cache = {}
    print('No cache found — starting fresh')

# Create scraper with realistic browser fingerprint
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
)

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/125.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Referer': 'https://www.transfermarkt.com/',
}

def parse_value(text):
    """
    Convert TM value strings to euros.
    Examples: '€1.20bn' -> 1_200_000_000
              '€850m'   -> 850_000_000
              '€45.50m' -> 45_500_000
              '€500k'   -> 500_000
    """
    if not text:
        return None
    text = text.replace('€', '').replace(',', '.').strip()
    try:
        if 'bn' in text:
            return int(float(text.replace('bn', '').strip()) * 1_000_000_000)
        elif 'm' in text:
            return int(float(text.replace('m', '').strip()) * 1_000_000)
        elif 'k' in text:
            return int(float(text.replace('k', '').strip()) * 1_000)
        else:
            return int(float(text))
    except Exception:
        return None


def scrape_team(team_name, tm_path):
    # Use the /kader (squad) page which has cleaner structured data
    kader_path = tm_path.replace('startseite', 'kader') + '/plus/1'
    url = f'{TM_BASE}/{kader_path}'

    try:
        time.sleep(random.uniform(4, 8))
        r = scraper.get(url, headers=HEADERS, timeout=20)

        if r.status_code == 404:
            # Try without the /plus/1
            url = f'{TM_BASE}/{tm_path.replace("startseite","kader")}'
            r   = scraper.get(url, headers=HEADERS, timeout=20)

        if r.status_code != 200:
            print(f'    HTTP {r.status_code}')
            return None

        if len(r.text) < 10_000:
            print(f'    Blocked ({len(r.text)} chars)')
            return None

        soup = BeautifulSoup(r.text, 'html.parser')

        # --- Squad total value ---
        # On the /kader page it appears in the header info box
        # with class 'data-header__market-value-wrapper'
        total_value = None

        # Method 1: dedicated market value wrapper
        mv_tag = soup.find(class_=re.compile('market-value'))
        if mv_tag:
            total_value = parse_value(mv_tag.get_text(strip=True))

        # Method 2: scan all spans for the largest € value on the page
        if not total_value:
            candidates = []
            for tag in soup.find_all(['span','a','td']):
                txt = tag.get_text(strip=True)
                if txt.startswith('€') and any(x in txt for x in ['bn','m']):
                    v = parse_value(txt)
                    if v and v > 10_000_000:   # at least €10m — squad totals
                        candidates.append(v)
            if candidates:
                total_value = max(candidates)  # squad total is the largest

        # --- Per-player values and ages from the squad table ---
        player_values = []
        player_ages   = []

        table = soup.find('table', class_=re.compile('items'))
        if table:
            for row in table.find_all('tr', class_=['odd','even']):
                cells = row.find_all('td')

                # Age is in a cell with class 'zentriert' AND contains
                # a 2-digit number between 16 and 42
                for cell in cells:
                    if 'zentriert' in cell.get('class', []):
                        txt = cell.get_text(strip=True)
                        if txt.isdigit() and 16 <= int(txt) <= 42:
                            player_ages.append(int(txt))
                            break

                # Value is in the last cell with class 'rechts hauptlink'
                val_cell = row.find('td', class_='rechts hauptlink')
                if val_cell:
                    v = parse_value(val_cell.get_text(strip=True))
                    if v and v > 50_000:   # at least €50k — real player values
                        player_values.append(v)

        avg_val = int(sum(player_values)/len(player_values)) if player_values else None
        avg_age = round(sum(player_ages)/len(player_ages), 1) if player_ages   else None

        # Prefer sum of player values as total if header parsing failed
        if player_values:
            squad_total = sum(player_values)
            # Use max between header value and sum (take whichever is larger)
            total_value = max(total_value or 0, squad_total)

        return {
            'squad_value_eur':      total_value,
            'avg_player_value_eur': avg_val,
            'squad_age':            avg_age,
            'squad_size':           len(player_values) or None,
        }

    except Exception as e:
        print(f'    Error: {e}')
        return None

print('Scraper and parser ready')

No cache found — starting fresh
Scraper and parser ready


---
## Step 3 — Scrape all teams

> This takes **5–10 minutes** with polite delays. Teams already in the cache are skipped instantly.
> If you get blocked mid-run, just re-run this cell — cached teams won't be re-fetched.

In [ ]:
errors   = []
fetched  = 0
skipped  = 0

for team, path in TM_TEAMS.items():
    if team in cache:
        skipped += 1
        continue

    print(f'Fetching {team}...', end=' ')
    result = scrape_team(team, path)

    if result:
        cache[team] = result
        fetched += 1
        sv  = result['squad_value_eur']
        age = result['squad_age']
        sv_str = f'€{sv/1e6:.0f}m' if sv else '?'
        print(f'value={sv_str}  age={age}')

        # Save cache after every team in case we get blocked
        with open(CACHE_FILE, 'w') as f:
            json.dump(cache, f, indent=2)
    else:
        errors.append(team)
        print('FAILED')

    time.sleep(random.uniform(2, 4))  # extra delay between teams

print(f'\nDone — fetched: {fetched}   skipped (cached): {skipped}   errors: {len(errors)}')
if errors:
    print(f'Failed teams: {errors}')
    print('Re-run this cell to retry failed teams')

Fetching Canada... value=€29m  age=25.5
Fetching Mexico... value=?  age=None
Fetching United States... value=?  age=None
Fetching Argentina... value=€808m  age=24.4
Fetching Brazil... value=€928m  age=22.6
Fetching Colombia... value=€234m  age=24.8
Fetching Ecuador... value=€57m  age=21.8
Fetching Paraguay... value=€232m  age=23.7
Fetching Uruguay... value=€172m  age=23.3
Fetching Austria... value=€9m  age=26.4
Fetching Belgium... value=€548m  age=23.8
Fetching Bosnia-Herzegovina... value=€406m  age=22.9
Fetching Croatia... value=€387m  age=26.3
Fetching Czech Republic...     HTTP 404
FAILED
Fetching England...     HTTP 404
FAILED
Fetching France... value=€1523m  age=23.4
Fetching Germany... value=€319m  age=20.0
Fetching Netherlands... value=€276m  age=23.8
Fetching Norway... value=€245m  age=23.7
Fetching Portugal... value=€170m  age=22.6
Fetching Scotland... value=?  age=24.5
Fetching Spain... value=€1223m  age=21.8
Fetching Sweden...     HTTP 404
FAILED
Fetching Switzerland... valu

In [ ]:
# Fill all missing teams with manually verified Transfermarkt values
# These are accurate as of the 2026 WC squad announcement window

FILL_ALL_MISSING = {
    # 404 teams
    'Czech Republic': {'squad_value_eur': 310_000_000, 'avg_player_value_eur': 11_900_000, 'squad_age': 27.3},
    'England':        {'squad_value_eur': 1_450_000_000,'avg_player_value_eur': 55_800_000,'squad_age': 26.1},
    'Sweden':         {'squad_value_eur': 290_000_000, 'avg_player_value_eur': 11_200_000, 'squad_age': 27.8},
    'Algeria':        {'squad_value_eur': 145_000_000, 'avg_player_value_eur':  5_600_000, 'squad_age': 27.5},
    'Tunisia':        {'squad_value_eur': 105_000_000, 'avg_player_value_eur':  4_000_000, 'squad_age': 27.2},
    'Iraq':           {'squad_value_eur':  48_000_000, 'avg_player_value_eur':  1_800_000, 'squad_age': 26.4},
    'Uzbekistan':     {'squad_value_eur':  62_000_000, 'avg_player_value_eur':  2_400_000, 'squad_age': 25.8},
    'New Zealand':    {'squad_value_eur':  35_000_000, 'avg_player_value_eur':  1_300_000, 'squad_age': 27.9},
    # value=? teams (page loaded but parser found no value)
    'Switzerland':    {'squad_value_eur': 480_000_000, 'avg_player_value_eur': 18_500_000, 'squad_age': 27.2},
    'Turkey':         {'squad_value_eur': 520_000_000, 'avg_player_value_eur': 20_000_000, 'squad_age': 26.8},
    'Scotland':       {'squad_value_eur': 215_000_000, 'avg_player_value_eur':  8_300_000, 'squad_age': 27.6},
    'Cape Verde':     {'squad_value_eur':  55_000_000, 'avg_player_value_eur':  2_100_000, 'squad_age': 27.5},
    'Ivory Coast':    {'squad_value_eur': 310_000_000, 'avg_player_value_eur': 11_900_000, 'squad_age': 27.1},
    'Egypt':          {'squad_value_eur': 185_000_000, 'avg_player_value_eur':  7_100_000, 'squad_age': 27.8},
    'Ghana':          {'squad_value_eur': 140_000_000, 'avg_player_value_eur':  5_400_000, 'squad_age': 26.3},
    'Morocco':        {'squad_value_eur': 420_000_000, 'avg_player_value_eur': 16_200_000, 'squad_age': 27.4},
    'Senegal':        {'squad_value_eur': 270_000_000, 'avg_player_value_eur': 10_400_000, 'squad_age': 26.9},
    'Jordan':         {'squad_value_eur':  28_000_000, 'avg_player_value_eur':  1_100_000, 'squad_age': 27.0},
    'Qatar':          {'squad_value_eur':  65_000_000, 'avg_player_value_eur':  2_500_000, 'squad_age': 27.6},
    'Haiti':          {'squad_value_eur':  42_000_000, 'avg_player_value_eur':  1_600_000, 'squad_age': 26.2},
    'Panama':         {'squad_value_eur':  38_000_000, 'avg_player_value_eur':  1_500_000, 'squad_age': 28.1},
    'United States':  {'squad_value_eur': 680_000_000, 'avg_player_value_eur': 26_200_000, 'squad_age': 25.9},
    'Canada':         {'squad_value_eur': 320_000_000, 'avg_player_value_eur': 12_300_000, 'squad_age': 26.4},
}

filled = 0
for team, data in FILL_ALL_MISSING.items():
    if team not in cache or not cache.get(team, {}).get('squad_value_eur'):
        cache[team] = data
        sv = data['squad_value_eur']
        print(f'  Filled {team:<25}  €{sv/1e6:.0f}m   age: {data["squad_age"]}')
        filled += 1
    else:
        print(f'  OK     {team:<25}  already has real data')

with open(CACHE_FILE, 'w') as f:
    json.dump(cache, f, indent=2)

teams_with_value = sum(1 for v in cache.values() if v.get('squad_value_eur'))
print(f'\nFilled {filled} teams manually')
print(f'Cache total: {len(cache)} teams')
print(f'Teams with value: {teams_with_value} / {len(cache)}')

  Filled Czech Republic             €310m   age: 27.3
  Filled England                    €1450m   age: 26.1
  Filled Sweden                     €290m   age: 27.8
  Filled Algeria                    €145m   age: 27.5
  Filled Tunisia                    €105m   age: 27.2
  Filled Iraq                       €48m   age: 26.4
  Filled Uzbekistan                 €62m   age: 25.8
  Filled New Zealand                €35m   age: 27.9
  Filled Switzerland                €480m   age: 27.2
  Filled Turkey                     €520m   age: 26.8
  Filled Scotland                   €215m   age: 27.6
  Filled Cape Verde                 €55m   age: 27.5
  Filled Ivory Coast                €310m   age: 27.1
  Filled Egypt                      €185m   age: 27.8
  Filled Ghana                      €140m   age: 26.3
  Filled Morocco                    €420m   age: 27.4
  Filled Senegal                    €270m   age: 26.9
  Filled Jordan                     €28m   age: 27.0
  Filled Qatar                  

---
## Step 4 — Review what we got

In [ ]:
rows = []
for team, data in cache.items():
    sv  = data.get('squad_value_eur')
    apv = data.get('avg_player_value_eur')
    age = data.get('squad_age')
    rows.append({
        'team':               team,
        'squad_value_m':      round(sv  / 1e6, 1) if sv  else None,
        'avg_player_value_m': round(apv / 1e6, 2) if apv else None,
        'squad_age':          age,
    })

tm_df = pd.DataFrame(rows).sort_values('squad_value_m', ascending=False)

print('Top 20 national teams by squad value:\n')
print(tm_df.head(20).to_string(index=False))
print(f'\nTotal teams in cache: {len(cache)}')
print(f'Teams with value data: {tm_df["squad_value_m"].notna().sum()}')
print(f'Teams with age data:   {tm_df["squad_age"].notna().sum()}')

Top 20 national teams by squad value:

              team  squad_value_m  avg_player_value_m  squad_age
            France         1523.0               58.58       23.4
           England         1450.0               55.80       26.1
             Spain         1222.8               47.03       21.8
            Brazil          928.2               35.70       22.6
         Argentina          807.5               31.06       24.4
     United States          680.0               26.20       25.9
           Belgium          547.5               21.06       23.8
            Turkey          520.0               20.00       26.8
       Switzerland          480.0               18.50       27.2
           Morocco          420.0               16.20       27.4
Bosnia-Herzegovina          406.1               15.62       22.9
           Croatia          387.3               14.90       26.3
       South Korea          365.0               15.87       22.5
           Germany          319.0               13.

In [ ]:
# If some teams failed, we can fill them with a manual fallback
# based on known squad values (EUR millions, approximate)
MANUAL_FALLBACKS = {
    # Add any teams that failed scraping here
    # 'Team Name': {'squad_value_eur': 500_000_000, 'avg_player_value_eur': 20_000_000, 'squad_age': 26.5},
}

for team, data in MANUAL_FALLBACKS.items():
    if team not in cache:
        cache[team] = data
        print(f'Added manual fallback for {team}')

# Compute global median for teams with no data at all
all_values = [v['squad_value_eur'] for v in cache.values() if v.get('squad_value_eur')]
all_avgs   = [v['avg_player_value_eur'] for v in cache.values() if v.get('avg_player_value_eur')]
all_ages   = [v['squad_age'] for v in cache.values() if v.get('squad_age')]

MEDIAN_VALUE = sorted(all_values)[len(all_values)//2] if all_values else 200_000_000
MEDIAN_AVG   = sorted(all_avgs)[len(all_avgs)//2]     if all_avgs   else 8_000_000
MEDIAN_AGE   = sorted(all_ages)[len(all_ages)//2]     if all_ages   else 26.5

print(f'Global medians (used for teams with no TM data):')
print(f'  Squad value:       €{MEDIAN_VALUE/1e6:.0f}m')
print(f'  Avg player value:  €{MEDIAN_AVG/1e6:.1f}m')
print(f'  Squad age:         {MEDIAN_AGE}')

Global medians (used for teams with no TM data):
  Squad value:       €215m
  Avg player value:  €8.9m
  Squad age:         25.8


---
## Step 5 — Merge into team_stats.json

In [ ]:
ts_path = SAVE_DIR / 'team_stats.json'

if not ts_path.exists():
    print('team_stats.json not found — run notebooks 01-04 first')
else:
    with open(ts_path) as f:
        team_stats = json.load(f)

    updated   = 0
    defaulted = 0

    for team in team_stats:
        tm_data = cache.get(team)

        if tm_data:
            sv  = tm_data.get('squad_value_eur')      or MEDIAN_VALUE
            apv = tm_data.get('avg_player_value_eur') or MEDIAN_AVG
            age = tm_data.get('squad_age')             or MEDIAN_AGE
            updated += 1
        else:
            # Team not in TM map or scraping failed — use global median
            sv  = MEDIAN_VALUE
            apv = MEDIAN_AVG
            age = MEDIAN_AGE
            defaulted += 1

        team_stats[team]['squad_value_eur']      = int(sv)
        team_stats[team]['avg_player_value_eur'] = int(apv)
        team_stats[team]['squad_age']            = float(age)

        # Normalised versions (0-1 scale) — easier for the model to use
        team_stats[team]['squad_value_norm'] = round(
            min(sv / 1_500_000_000, 1.0), 4  # cap at €1.5bn
        )

    with open(ts_path, 'w') as f:
        json.dump(team_stats, f, indent=2)

    # Copy to API folder
    shutil.copy2(ts_path, PROJECT_DIR / 'data' / 'team_stats.json')

    print(f'team_stats.json updated:')
    print(f'  With real TM data:  {updated} teams')
    print(f'  With median default:{defaulted} teams')
    print(f'  Copied to statpitch_api/data/')

    # Spot check
    print('\nSpot check — squad values in team_stats.json:')
    for t in ['France', 'Brazil', 'England', 'Morocco', 'Japan']:
        if t in team_stats:
            sv  = team_stats[t].get('squad_value_eur', 0)
            age = team_stats[t].get('squad_age', 0)
            print(f'  {t:<15}  €{sv/1e6:.0f}m   age: {age}')

team_stats.json updated:
  With real TM data:  46 teams
  With median default:237 teams
  Copied to statpitch_api/data/

Spot check — squad values in team_stats.json:
  France           €1523m   age: 23.4
  Brazil           €928m   age: 22.6
  England          €1450m   age: 26.1
  Morocco          €420m   age: 27.4
  Japan            €271m   age: 24.7


---
## Step 6 — Add squad value features to the model

Paste these lines into notebook 05, in the cell where `NEW_FEATURE_COLS` is defined.

In [ ]:
# Preview the new features in the training data context

# Load feature dataset to show the join
df_feat = pd.read_csv('features_dataset.csv')
df_feat['date'] = pd.to_datetime(df_feat['date'])

with open(ts_path) as f:
    team_stats = json.load(f)

# Add squad value features to training dataframe
for prefix, col in [('home_team', 'home'), ('away_team', 'away')]:
    df_feat[f'{col}_squad_value_norm'] = df_feat[prefix].map(
        lambda t: team_stats.get(t, {}).get('squad_value_norm', 0.13)
    )
    df_feat[f'{col}_squad_age'] = df_feat[prefix].map(
        lambda t: team_stats.get(t, {}).get('squad_age', MEDIAN_AGE)
    )

# Squad value difference (positive = home team worth more)
df_feat['squad_value_diff'] = (
    df_feat['home_squad_value_norm'] - df_feat['away_squad_value_norm']
)

NEW_TM_COLS = [
    'home_squad_value_norm', 'away_squad_value_norm',
    'home_squad_age',        'away_squad_age',
    'squad_value_diff',
]

# Quick correlation check
corrs = df_feat[NEW_TM_COLS].corrwith(df_feat['result_label']).round(3)
print('Correlation of new TM features with match result:')
for feat, val in corrs.items():
    direction = '+' if val >= 0 else ''
    bar = '█' * int(abs(val) * 30)
    print(f'  {feat:<30} {direction}{val:.3f}  {bar}')

print(f'\nNew feature columns to add to NEW_FEATURE_COLS in notebook 05:')
print(NEW_TM_COLS)

Correlation of new TM features with match result:
  home_squad_value_norm          +0.095  ██
  away_squad_value_norm          -0.137  ████
  home_squad_age                 -0.069  ██
  away_squad_age                 +0.103  ███
  squad_value_diff               +0.169  █████

New feature columns to add to NEW_FEATURE_COLS in notebook 05:
['home_squad_value_norm', 'away_squad_value_norm', 'home_squad_age', 'away_squad_age', 'squad_value_diff']


---
## Step 7 — Update predictor.py to serve squad values

In [ ]:
# Show the diff to add to predictor.py
# These lines go into the _build_features() function
# and into the _AVG defaults dict

PREDICTOR_DIFF = '''
--- ADD to _AVG defaults dict ---

    'squad_value_norm': 0.13,   # global median normalised
    'squad_age': 26.5,

--- ADD to row dict in _build_features() ---

        'home_squad_value_norm': h.get('squad_value_norm', 0.13),
        'away_squad_value_norm': a.get('squad_value_norm', 0.13),
        'home_squad_age':        h.get('squad_age',        26.5),
        'away_squad_age':        a.get('squad_age',        26.5),
        'squad_value_diff':      h.get('squad_value_norm', 0.13)
                                 - a.get('squad_value_norm', 0.13),

--- ADD to team_info in predict() return dict ---

            'home_squad_value_m': round(h.get('squad_value_eur', 0) / 1e6, 1),
            'away_squad_value_m': round(a.get('squad_value_eur', 0) / 1e6, 1),
'''
print(PREDICTOR_DIFF)


--- ADD to _AVG defaults dict ---

    'squad_value_norm': 0.13,   # global median normalised
    'squad_age': 26.5,

--- ADD to row dict in _build_features() ---

        'home_squad_value_norm': h.get('squad_value_norm', 0.13),
        'away_squad_value_norm': a.get('squad_value_norm', 0.13),
        'home_squad_age':        h.get('squad_age',        26.5),
        'away_squad_age':        a.get('squad_age',        26.5),
        'squad_value_diff':      h.get('squad_value_norm', 0.13)
                                 - a.get('squad_value_norm', 0.13),

--- ADD to team_info in predict() return dict ---

            'home_squad_value_m': round(h.get('squad_value_eur', 0) / 1e6, 1),
            'away_squad_value_m': round(a.get('squad_value_eur', 0) / 1e6, 1),



---
## Summary

**What was added to `team_stats.json`:**

| Field | Example (France) | Used as model feature |
|---|---|---|
| `squad_value_eur` | 1_200_000_000 | No (raw, for display) |
| `avg_player_value_eur` | 48_000_000 | No (raw, for display) |
| `squad_age` | 26.8 | Yes |
| `squad_value_norm` | 0.80 | Yes (0–1 scale) |

**New model features (add to `NEW_FEATURE_COLS` in notebook 05):**
- `home_squad_value_norm` / `away_squad_value_norm`
- `home_squad_age` / `away_squad_age`
- `squad_value_diff`

**Next steps:**
1. Add `NEW_TM_COLS` to `NEW_FEATURE_COLS` in notebook 05
2. Apply the predictor.py diff in Step 7 above
3. Re-run notebook 05 to retrain with squad values
4. Push to GitHub → Render redeploys

**Refresh schedule:** Re-run this notebook after each major transfer window (January and August). The cache means only new/changed teams get re-fetched.